In [12]:
import pandas as pd
import numpy as np
import difflib

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr, spearmanr

import evaluate

############################################
# CONFIG
############################################

UHM_PATH  = "../data/uh-mazing.csv"
GOLD_PATH = "../data/uh-mazing-gold-irr.csv"

ID_COL = "ID"

LANGS = ["ZH", "ES", "HI", "FR", "DE", "IT", "CS", "AR"]

# POLICY:
# uh-mazing → prefer *_disfluent, else LANG
UHM_PRIORITY_SUFFIXES = ["_disfluent", ""]

OUT_PER_ITEM = "uhmazing_vs_gold_per_item.csv"
OUT_SUMMARY  = "uhmazing_vs_gold_summary.csv"

############################################
# HELPERS
############################################

def teleki_diff(a: str, b: str) -> float:
    """Teleki et al. string disagreement."""
    return difflib.SequenceMatcher(None, a, b).ratio()

def pick_uhm_col(cols, lang):
    """Prefer *_disfluent over bare LANG."""
    for suf in UHM_PRIORITY_SUFFIXES:
        name = f"{lang}{suf}"
        if name in cols:
            return name
    return None

def pick_gold_col(cols, lang):
    """Gold file should ONLY have bare LANG."""
    return lang if lang in cols else None

def safe_corr(x, y, method="pearson"):
    if len(x) < 3:
        return np.nan
    try:
        if method == "pearson":
            return pearsonr(x, y)[0]
        return spearmanr(x, y)[0]
    except Exception:
        return np.nan

############################################
# LOAD + JOIN
############################################

uhm = pd.read_csv(UHM_PATH)
gold = pd.read_csv(GOLD_PATH)

if ID_COL not in uhm.columns or ID_COL not in gold.columns:
    raise ValueError("Both CSVs must contain an ID column")

merged = uhm.merge(gold, on=ID_COL, how="inner", suffixes=("_uhm", "_gold"))

############################################
# MODELS
############################################

embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

############################################
# METRIC COMPUTATION
############################################

rows = []

for lang in LANGS:
    uhm_col  = pick_uhm_col(merged.columns, lang)
    gold_col = pick_gold_col(merged.columns, lang)

    if uhm_col is None or gold_col is None:
        continue

    sub = merged[[ID_COL, uhm_col, gold_col]].copy()

    # Strict filtering (Series-safe, no index bugs)
    sub = sub[
        sub[uhm_col].notna()
        & sub[gold_col].notna()
        & sub[uhm_col].astype(str).str.strip().ne("").replace("_","")
        & sub[gold_col].astype(str).str.strip().ne("").replace("_","")
    ].copy()

    if sub.empty:
        continue

    a_texts = sub[uhm_col].astype(str).tolist()   # uh-mazing
    b_texts = sub[gold_col].astype(str).tolist()  # gold

    # Teleki IRR
    teleki = np.array([
        teleki_diff(a, b) for a, b in zip(a_texts, b_texts)
    ])

    # Embedding IRR
    a_emb = embedder.encode(a_texts, normalize_embeddings=True)
    b_emb = embedder.encode(b_texts, normalize_embeddings=True)
    emb_irr = cosine_similarity(a_emb, b_emb).diagonal()

    for rid, t, e in zip(
        sub[ID_COL], teleki, emb_irr
    ):
        rows.append({
            "ID": rid,
            "language": lang,
            "uhm_col": uhm_col,
            "gold_col": gold_col,
            "teleki_irr": float(t),
            "embedding_irr": float(e), 
        })

############################################
# OUTPUT
############################################

results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError("No rows produced — check ID overlap and language columns")

summary = (
    results
    .groupby("language")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "teleki_irr_mean": g["teleki_irr"].mean(),
        "embedding_irr_mean": g["embedding_irr"].mean(),
        "teleki_vs_embedding_pearson":
            safe_corr(g["teleki_irr"], g["embedding_irr"], "pearson"),
        "teleki_vs_embedding_spearman":
            safe_corr(g["teleki_irr"], g["embedding_irr"], "spearman"),
    }))
    .reset_index()
)

results.to_csv(OUT_PER_ITEM, index=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("Wrote:", OUT_PER_ITEM)
print("Wrote:", OUT_SUMMARY)
print("\n=== SUMMARY ===")
display(summary.round(2))
display(summary.describe().round(2))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1728.75it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wrote: uhmazing_vs_gold_per_item.csv
Wrote: uhmazing_vs_gold_summary.csv

=== SUMMARY ===


/var/folders/n2/yfmlyvts65xd64jgl_j_5jk80000gn/T/ipykernel_57316/1421148218.py:141: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,language,n,teleki_irr_mean,embedding_irr_mean,teleki_vs_embedding_pearson,teleki_vs_embedding_spearman
0,AR,10.0,0.51,0.94,0.30,0.09
1,CS,10.0,0.38,0.93,0.16,0.12
2,DE,10.0,0.47,0.91,-0.15,-0.31
3,ES,10.0,0.34,0.78,0.72,0.81
4,FR,10.0,0.38,0.89,0.27,0.10
5,HI,10.0,0.65,0.92,0.62,0.49
6,IT,10.0,0.53,0.93,0.24,-0.01
7,ZH,10.0,0.44,0.91,0.59,0.55


,n,teleki_irr_mean,embedding_irr_mean,teleki_vs_embedding_pearson,teleki_vs_embedding_spearman
count,8.0,8.00,8.00,8.00,8.00
mean,10.0,0.46,0.90,0.35,0.23
std,0.0,0.10,0.05,0.29,0.36
min,10.0,0.34,0.78,-0.15,-0.31
25%,10.0,0.38,0.90,0.22,0.07
50%,10.0,0.46,0.91,0.29,0.11
75%,10.0,0.52,0.93,0.60,0.51
max,10.0,0.65,0.94,0.72,0.81
